# Despliegue en Amazon SageMaker

Este notebook **no se ejecuta en el entorno local**: debe correrse desde una instancia de
notebook de SageMaker, SageMaker Studio, o cualquier entorno con credenciales de AWS y el
SDK `sagemaker` instalado (`pip install sagemaker boto3`).

Reutiliza el script `train.py` de esta misma carpeta, que contiene la arquitectura de la CNN
ganadora (3 bloques `Conv3x3 -> ReLU -> MaxPool`) ya entrenada y justificada en
`notebooks/EuroSAT_CNN.ipynb`.

**Advertencia de costos**: lanzar un training job y desplegar un endpoint genera cargos en la
cuenta de AWS mientras el endpoint esté activo. La última celda lo elimina; no lo omitas.

In [ ]:
import sagemaker
import boto3

session = sagemaker.Session()
role = sagemaker.get_execution_role()  # requiere ejecutarse dentro de SageMaker (Studio/notebook instance)
bucket = session.default_bucket()
region = session.boto_region_name

print(f"bucket: {bucket}")
print(f"region: {region}")

## 1. Subir el dataset a S3

Se asume que el dataset ya fue organizado localmente en dos carpetas con estructura
`ImageFolder` (una subcarpeta por clase), producidas a partir del split usado en el notebook
principal: `data/train/<clase>/*.jpg` y `data/val/<clase>/*.jpg`.

In [ ]:
train_s3 = session.upload_data(path="../data/train", bucket=bucket, key_prefix="eurosat/train")
val_s3 = session.upload_data(path="../data/val", bucket=bucket, key_prefix="eurosat/val")

print(train_s3)
print(val_s3)

## 2. Definir el Estimator de PyTorch

Se usa el contenedor administrado de PyTorch de SageMaker, apuntando a `train.py`. El tipo de
instancia `ml.m5.xlarge` es suficiente (CPU) dado el tamaño reducido del modelo; puede
cambiarse a una instancia con GPU (`ml.g4dn.xlarge`) si se desea acelerar el entrenamiento.

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    source_dir=".",
    role=role,
    framework_version="2.1",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    instance_count=1,
    hyperparameters={
        "epochs": 15,
        "batch-size": 64,
        "lr": 1e-3,
    },
)

## 3. Entrenar

`fit` bloquea hasta que termina el training job y sube los logs/métricas a CloudWatch.

In [ ]:
estimator.fit({"train": train_s3, "val": val_s3})

## 4. Desplegar el endpoint

Crea un endpoint HTTPS en tiempo real. `ml.t2.medium` es suficiente para inferencia de este
modelo pequeño.

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
)

## 5. Probar el endpoint

Se envía un tensor de ejemplo (una imagen normalizada de 64x64x3) y se revisa la predicción.

In [ ]:
import numpy as np

sample = np.random.rand(1, 3, 64, 64).astype("float32")  # reemplazar por una imagen real preprocesada
result = predictor.predict(sample)
print(result)

## 6. Limpieza (obligatorio)

El endpoint sigue generando costos mientras esté activo. Eliminarlo al terminar las pruebas.

In [ ]:
predictor.delete_endpoint()